# 🧠🩺 MedPredict AI - Interactive Analysis

AI-powered healthcare intelligence system for disease prediction and patient health data analysis.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')

print('✅ Libraries loaded successfully!')

## 1. Load and Explore Data

In [ ]:
from src.data_preprocessing import load_diabetes_data, load_heart_data, prepare_dataset, get_dataset_summary

# Load datasets
diabetes_df = load_diabetes_data()
heart_df = load_heart_data()

print('📊 Diabetes Dataset:')
print(f'   Shape: {diabetes_df.shape}')
print(f'   Missing: {diabetes_df.isnull().sum().sum()}')
display(diabetes_df.head())

print('\n📊 Heart Disease Dataset:')
print(f'   Shape: {heart_df.shape}')
print(f'   Missing: {heart_df.isnull().sum().sum()}')
display(heart_df.head())

In [ ]:
print('📈 Diabetes Dataset Statistics:')
display(diabetes_df.describe())

print('\n📈 Heart Disease Dataset Statistics:')
display(heart_df.describe())

## 2. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Diabetes
colors = ['#2ecc71', '#e74c3c']
diabetes_df['Outcome'].value_counts().plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Diabetes Target Distribution')
axes[0].set_xticklabels(['Non-Diabetic (0)', 'Diabetic (1)'], rotation=0)
axes[0].set_ylabel('Count')

# Heart Disease
heart_df['HeartDiseaseRisk'].value_counts().plot(kind='bar', ax=axes[1], color=colors, edgecolor='black')
axes[1].set_title('Heart Disease Target Distribution')
axes[1].set_xticklabels(['Low Risk (0)', 'High Risk (1)'], rotation=0)
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3. Correlation Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 9))

# Diabetes correlations
corr1 = diabetes_df.corr(numeric_only=True)
mask1 = np.triu(np.ones_like(corr1, dtype=bool))
sns.heatmap(corr1, mask=mask1, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[0])
axes[0].set_title('Diabetes - Correlation Heatmap')

# Heart disease correlations
corr2 = heart_df.corr(numeric_only=True)
mask2 = np.triu(np.ones_like(corr2, dtype=bool))
sns.heatmap(corr2, mask=mask2, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[1])
axes[1].set_title('Heart Disease - Correlation Heatmap')

plt.tight_layout()
plt.show()

## 4. Feature Analysis

In [ ]:
from src.feature_engineering import analyze_features, get_feature_importance

# Diabetes features
X_train_d, X_test_d, y_train_d, y_test_d, scaler_d, features_d = prepare_dataset(diabetes_df, 'Outcome')
importance_d = get_feature_importance(X_train_d, y_train_d, features_d)
analysis_d = analyze_features(X_train_d, y_train_d, features_d)

# Heart features
X_train_h, X_test_h, y_train_h, y_test_h, scaler_h, features_h = prepare_dataset(heart_df, 'HeartDiseaseRisk')
importance_h = get_feature_importance(X_train_h, y_train_h, features_h)
analysis_h = analyze_features(X_train_h, y_train_h, features_h)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=importance_d, x='importance', y='feature', palette='viridis', ax=axes[0])
axes[0].set_title('Diabetes - Feature Importance')

sns.barplot(data=importance_h, x='importance', y='feature', palette='viridis', ax=axes[1])
axes[1].set_title('Heart Disease - Feature Importance')

plt.tight_layout()
plt.show()

print('\n🧠 Diabetes - Top Features:')
display(analysis_d)

print('\n🧠 Heart Disease - Top Features:')
display(analysis_h)

## 5. Train & Evaluate Models

In [ ]:
from src.models import get_models, train_and_evaluate, get_best_model

models = get_models()
print(f'🔧 Training {len(models)} models on each dataset...\n')

# Diabetes
results_d = train_and_evaluate(models, X_train_d, X_test_d, y_train_d, y_test_d)
best_d = get_best_model(results_d)
print(f'🏆 Best Diabetes Model: {best_d["model"]} (ROC AUC: {best_d["roc_auc"]:.4f})')

# Heart Disease
models2 = get_models()  # fresh instances
results_h = train_and_evaluate(models2, X_train_h, X_test_h, y_train_h, y_test_h)
best_h = get_best_model(results_h)
print(f'🏆 Best Heart Disease Model: {best_h["model"]} (ROC AUC: {best_h["roc_auc"]:.4f})')

In [ ]:
# Results table
metrics_cols = ['model', 'accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']

print('📊 Diabetes Model Results:')
display(pd.DataFrame([{k: r[k] for k in metrics_cols} for r in results_d]).style.highlight_max(subset=metrics_cols[1:], color='lightgreen'))

print('\n📊 Heart Disease Model Results:')
display(pd.DataFrame([{k: r[k] for k in metrics_cols} for r in results_h]).style.highlight_max(subset=metrics_cols[1:], color='lightgreen'))

## 6. ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, auc

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for r in results_d:
    y_prob = r['trained_model'].predict_proba(X_test_d)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_d, y_prob)
    axes[0].plot(fpr, tpr, linewidth=2, label=f'{r["model"]} (AUC={auc(fpr, tpr):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_title('Diabetes - ROC Curves')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')

for r in results_h:
    y_prob = r['trained_model'].predict_proba(X_test_h)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_h, y_prob)
    axes[1].plot(fpr, tpr, linewidth=2, label=f'{r["model"]} (AUC={auc(fpr, tpr):.3f})')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_title('Heart Disease - ROC Curves')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

## 7. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(best_d['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'], ax=axes[0])
axes[0].set_title(f'Diabetes - {best_d["model"]}')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(best_h['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'], ax=axes[1])
axes[1].set_title(f'Heart Disease - {best_h["model"]}')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 8. Make Predictions

In [ ]:
from src.predict import predict_diabetes, predict_heart_disease, get_sample_diabetes_patient, get_sample_heart_patient

# Diabetes prediction
patient_d = get_sample_diabetes_patient()
result_d = predict_diabetes(best_d['trained_model'], scaler_d, patient_d)

print('🩺 Diabetes Prediction')
print(f'   Patient: {patient_d}')
print(f'   Result:  {result_d["label"]} (Confidence: {result_d["confidence"]:.1f}%)')
print(f'   P(Negative): {result_d["probability_negative"]:.1f}% | P(Positive): {result_d["probability_positive"]:.1f}%')

print()

# Heart disease prediction
patient_h = get_sample_heart_patient()
result_h = predict_heart_disease(best_h['trained_model'], scaler_h, patient_h)

print('❤️ Heart Disease Prediction')
print(f'   Patient: {patient_h}')
print(f'   Result:  {result_h["label"]} (Confidence: {result_h["confidence"]:.1f}%)')
print(f'   P(Low Risk): {result_h["probability_negative"]:.1f}% | P(High Risk): {result_h["probability_positive"]:.1f}%')

## 9. Custom Prediction

Modify the patient data below to make your own predictions.

In [ ]:
# ✏️ Edit this patient data for custom diabetes prediction
my_patient = {
    'Pregnancies': 1,
    'Glucose': 120,
    'BloodPressure': 70,
    'SkinThickness': 30,
    'Insulin': 80,
    'BMI': 25.0,
    'DiabetesPedigreeFunction': 0.5,
    'Age': 35,
}

result = predict_diabetes(best_d['trained_model'], scaler_d, my_patient)
print(f'🔮 Prediction: {result["label"]}')
print(f'📊 Confidence: {result["confidence"]:.1f}%')
print(f'   P(Non-Diabetic): {result["probability_negative"]:.1f}%')
print(f'   P(Diabetic):     {result["probability_positive"]:.1f}%')

---

⚠️ **Disclaimer**: This project is for educational and research purposes only and should not replace professional medical advice.